# Leçon 11 - Protocole de Contexte de Modèle (MCP)

Le **Protocole de Contexte de Modèle (MCP)** est une norme ouverte qui permet aux agents de découvrir et d'utiliser dynamiquement des outils, des ressources et des sources de données à l'exécution. Au lieu d'intégrer les outils directement dans un agent, MCP permet aux agents de se connecter à des serveurs externes qui exposent des capacités à la demande.

Dans cette leçon, vous apprendrez :
- Ce qu'est le MCP et pourquoi il est important pour les systèmes d'agents
- Comment fonctionne l'architecture client-serveur du MCP
- Comment construire des agents qui utilisent la découverte d'outils de type MCP


## Configuration

**Prérequis :**
- Projet Microsoft Foundry avec un modèle déployé
- Exécutez `az login` pour l'authentification


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Qu'est-ce que le protocole Model Context (MCP) ?

MCP définit une méthode standard permettant aux agents d'IA de découvrir et d'interagir avec des outils externes et des sources de données :

- **Serveur MCP** : Expose des outils, ressources et invites via un protocole standard
- **Client MCP** : Le runtime agent qui se connecte aux serveurs et découvre les capacités disponibles
- **Découverte dynamique** : Les agents n'ont pas besoin d'outils codés en dur — ils découvrent ce qui est disponible au moment de l'exécution

Ceci est puissant pour construire des systèmes d'agents extensibles où de nouvelles capacités peuvent être ajoutées sans modifier le code de l'agent.


## Comment fonctionne MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. L'agent (client MCP) se connecte à un serveur MCP
2. Le serveur répond avec une liste d'outils disponibles et leurs schémas
3. L'agent peut alors appeler n'importe quel outil découvert lors de son raisonnement
4. Les résultats circulent ensuite via le même protocole


## Simulation de la découverte d'outil MCP

Comme un vrai serveur MCP nécessite un processus serveur en cours d'exécution, nous allons démontrer le modèle en utilisant des fonctions `@tool` qui simulent ce qu'un service d'accommodation connecté MCP fournirait.

En production, ces outils seraient découverts dynamiquement à partir d'un serveur MCP plutôt que définis localement.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Construire un agent avec des outils de style MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP en production

Dans un environnement de production, MCP permet des schémas puissants :

- **Découverte dynamique des outils** : Les agents se connectent aux serveurs MCP et découvrent les outils à l'exécution
- **Architecture découplée** : Les fournisseurs d’outils peuvent mettre à jour indépendamment de l’agent
- **Partage inter-organisationnel** : Les équipes peuvent exposer des capacités via les serveurs MCP que n’importe quel agent peut utiliser
- **Support du Microsoft Agent Framework** : MAF a un support client MCP intégré via l’intégration `mcp`

Pour utiliser un serveur MCP réel avec MAF, vous vous connecteriez via `hosted_mcp_tool()` ou l’intégration client MCP.

**En savoir plus :**
- [Spécification MCP](https://modelcontextprotocol.io/)
- [Support MCP du Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Résumé

Dans cette leçon, vous avez appris :
- **MCP** est une norme ouverte pour la découverte dynamique d'outils entre agents et fournisseurs d'outils
- L'**architecture client-serveur** permet aux agents de découvrir les capacités au moment de l'exécution
- MCP permet des **systèmes d'agents extensibles et découplés** où des outils peuvent être ajoutés sans modifications de code
- Microsoft Agent Framework fournit un **support MCP intégré** pour une utilisation en production


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
